# 02 - Train the forward hi->en Transformer (RoPE)

Thin driver over `nmt.train.loop.Trainer`. Trains on the corpus + tokenizer built by `01_data`.

**Now configured for the RoPE variant:** `ModelConfig(pos_encoding="rope")`, with checkpoints + TensorBoard logs going to a **separate** Drive folder `checkpoints/forwardtranslate_rope` so the sinusoidal `forwardtranslate/best.pt` (BLEU 20.07) isn't clobbered. RoPE changes weight semantics, so this is a **fresh** run — the sinusoidal `best.pt` can't transfer. The loop **auto-resumes** from the latest `step_*.pt` if you reconnect. Patience-based **early stopping** halts when dev loss plateaus, so `max_steps` is just an upper bound.

**Before running:** GPU runtime (Runtime -> Change runtime type -> GPU), and `01_data` must have populated `DATA_ROOT` on Drive (`train.labse.*`, `dev.clean.*`, `spm_unigram_16000.model`).

The loop reports dev **loss** (nll); for **BLEU** run `04_decode_eval` (also set to `pos_encoding="rope"`) against the `best.pt` this writes to Drive.

## 1. Repo + install

In [1]:
import os, sys

REPO_URL = "https://github.com/Rishi-Jain-27/hindi-translator.git"
REPO_DIR = "/content/hindi-translator"

!test -d {REPO_DIR} && git -C {REPO_DIR} pull || git clone {REPO_URL} {REPO_DIR}
os.chdir(REPO_DIR)
sys.path.insert(0, os.path.abspath("src"))
print("cwd:", os.getcwd())

Cloning into '/content/hindi-translator'...
remote: Enumerating objects: 399, done.
remote: Counting objects: 100% (399/399), done.
remote: Compressing objects: 100% (235/235), done.
remote: Total 399 (delta 194), reused 333 (delta 128), pack-reused 0 (from 0)
Receiving objects: 100% (399/399), 13.79 MiB | 20.52 MiB/s, done.
Resolving deltas: 100% (194/194), done.
cwd: /content/hindi-translator


In [2]:
!pip install -e .

Obtaining file:///content/hindi-translator
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 147.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.1/121.1 kB 14.9 MB/s eta 0:00:00
  Building editable for nmt (pyproject.toml) ... done
  Created wheel for nmt: filename=nmt-0.1.0-0.editable-py3-none-any.whl size=1279 sha256=23d0cf251e1830c86c13307617766d8b0a688b7ca955cd2b5158a845eebde93f
  Stored in directory: /tmp/pip-ephem-wheel-cache-sxu1px6w/wheels/bd/c2/ef/ae818ff943d77ea8d63ef07aea61a1b82808716362dc169d4c
Successfully built nmt


In [3]:
import torch
print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

cuda: True NVIDIA A100-SXM4-80GB


## 2. Configs

`ModelConfig` is the base Transformer (d_model 512, 8 heads, 6+6 layers). `TrainConfig` below uses **real-run** values; the inline comments show the smoke values to drop back to for a quick test.

In [ ]:
from nmt.config import DataConfig, ModelConfig, TrainConfig
from google.colab import drive

# read the corpus + tokenizer that 01_data wrote to Drive (survives runtime resets)
drive.mount("/content/drive")
DATA_ROOT = "/content/drive/MyDrive/hindi-translator/data"
CKPT_ROOT = "/content/drive/MyDrive/hindi-translator/checkpoints/forwardtranslate_rope"   # RoPE forward hi->en ckpts; SEPARATE from sinusoidal forwardtranslate/ so its best.pt (BLEU 20.07) isn't clobbered

data_cfg = DataConfig(raw_dir=f"{DATA_ROOT}/raw", cache_dir=f"{DATA_ROOT}/cache")
model_cfg = ModelConfig(pos_encoding="rope")   # base + RoPE; FRESH run (sinusoidal best.pt can't transfer). 04_decode_eval must set pos_encoding="rope" to match

train_cfg = TrainConfig(
    out_dir=CKPT_ROOT,         # REAL RUN: checkpoints + TB logs -> Drive, survive a disconnect (loop auto-resumes)
    # real-run schedule (smoke values were max_steps=300, warmup=100, log=20, eval=100, ckpt=100)
    max_steps=100_000,         # upper bound; patience-based early stopping usually halts sooner
    warmup_steps=4000,         # standard Transformer inverse-sqrt warmup
    eval_every=2000,           # dev-nll eval cadence
    ckpt_every=2000,           # ~600 MB/ckpt to Drive; rotation keeps the last 2 (best.pt always kept)
    log_every=50,
)
print("out_dir:", train_cfg.out_dir)
print("pos_encoding:", model_cfg.pos_encoding)
print("eff tokens/step:", train_cfg.max_tokens * train_cfg.grad_accum,
      "| amp:", train_cfg.amp, train_cfg.amp_dtype, "| max_steps:", train_cfg.max_steps)

## 3. Tokenizer + data loaders

Train on the LaBSE-filtered train; evaluate on the cleaned dev set. The token-batch loader packs ~`max_tokens` tokens (incl. padding) per micro-batch.

In [5]:
from nmt.data.tokenizer import Tokenizer
from nmt.data.dataset import make_dataloader

cache = data_cfg.cache_dir
tok = Tokenizer.load(os.path.join(cache, f"spm_{data_cfg.tokenizer_model}_{data_cfg.vocab_size}.model"))
print("tokenizer vocab:", tok.vocab_size)

train_loader = make_dataloader(data_cfg, tok,
    os.path.join(cache, "train.labse.hi"), os.path.join(cache, "train.labse.en"),
    max_tokens=train_cfg.max_tokens, train=True)
dev_loader = make_dataloader(data_cfg, tok,
    os.path.join(cache, "dev.clean.hi"), os.path.join(cache, "dev.clean.en"),
    max_tokens=train_cfg.max_tokens, train=False)
print("train batches:", len(train_loader), "| dev batches:", len(dev_loader))

tokenizer vocab: 16000
train batches: 2691 | dev batches: 3


## 4. Build the model

In [6]:
from nmt.model.transformer import build_model

model = build_model(model_cfg, tok)     # copies vocab_size + special ids from the tokenizer
n_params = sum(p.numel() for p in model.parameters())
print(f"params: {n_params/1e6:.1f}M | d_model={model_cfg.d_model} heads={model_cfg.n_heads} "
      f"layers={model_cfg.n_enc_layers}+{model_cfg.n_dec_layers} tied={model_cfg.tie_embeddings}")

params: 52.3M | d_model=512 heads=8 layers=6+6 tied=True


## 5. (Optional) Live TensorBoard

Run this before training to watch `train/loss`, `train/lr`, `train/grad_norm`, `train/tok_per_s`, and `val/nll` update live.

In [ ]:
# TB event files are written under out_dir (now on Drive); live updates off a Drive mount can lag a bit
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/hindi-translator/checkpoints/forwardtranslate_rope

## 6. Train

Auto-resumes if `step_*.pt` already exist in the Drive `out_dir`. Watch `train/loss` fall from ~log(16000)=9.7 (random) downward and `val/nll` track down. Saves `best.pt` (best dev) + rolling `step_*.pt` to Drive. A disconnect is fine - reconnect and re-run to resume.

In [8]:
from nmt.train.loop import Trainer

trainer = Trainer(train_cfg, model, train_loader, dev_loader, tok, out_dir=CKPT_ROOT)
trainer.train()
print("done. step:", trainer.step, "| best dev nll:", trainer.best)

done. step: 36000 | best dev nll: 1.9370505617336886


## 7. Next

Once dev loss has plateaued (or early stopping fires), this run's `best.pt` is on Drive at `CKPT_ROOT`. Run **`04_decode_eval`** pointed at it for BLEU / chrF++ / TER on the test set. (Next build step before that: the KV-cache, to make decoding fast.)